# Logistic Regression

In [2]:
import pandas as pd
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, f1_score


features = ['danceability', 'energy', 'loudness', 'speechiness', 'acousticness',
            'instrumentalness', 'liveness', 'valence', 'tempo', 'duration_ms']

train = pd.read_csv('data/splits/train.csv')
val = pd.read_csv('data/splits/val.csv')
test = pd.read_csv('data/splits/test.csv')

X_train, y_train = train[features], train['genre_group']
X_val, y_val = val[features], val['genre_group']
X_test, y_test = test[features], test['genre_group']

print(len(train), len(val), len(test))  # should be roughly 80 / 10 / 10

35896 35896 17948


### using 'balanced' class weight

In [6]:
# scaling training data
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_val_s = scaler.transform(X_val)           # reuse train's mean/std
X_test_s = scaler.transform(X_test)

model = LogisticRegression(class_weight='balanced', max_iter=2000)
model.fit(X_train_s, y_train)

pred = model.predict(X_val_s)
print(classification_report(y_val, pred))

                      precision    recall  f1-score   support

chill_jazz_classical       0.57      0.43      0.49      4152
             country       0.17      0.51      0.25      1170
    electronic_dance       0.62      0.47      0.53      7728
          hiphop_rnb       0.11      0.14      0.12      1541
          indie_folk       0.11      0.36      0.16      1001
                kids       0.30      0.40      0.34      1579
               latin       0.44      0.39      0.42      3670
                 pop       0.22      0.05      0.09      4387
              reggae       0.07      0.45      0.13       776
          rock_metal       0.52      0.47      0.49      6737
               world       0.19      0.05      0.08      3155

            accuracy                           0.35     35896
           macro avg       0.30      0.34      0.28     35896
        weighted avg       0.41      0.35      0.36     35896



### using unbalanced class weights

In [13]:
# scaling training data
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_val_s = scaler.transform(X_val)           # reuse train's mean/std
X_test_s = scaler.transform(X_test)

model = LogisticRegression(class_weight=None, max_iter=2000)
model.fit(X_train_s, y_train)

pred = model.predict(X_val_s)
print(classification_report(y_val, pred))

                      precision    recall  f1-score   support

chill_jazz_classical       0.44      0.54      0.49      4152
             country       0.31      0.14      0.19      1170
    electronic_dance       0.45      0.69      0.55      7728
          hiphop_rnb       0.29      0.02      0.03      1541
          indie_folk       0.14      0.00      0.00      1001
                kids       0.50      0.30      0.38      1579
               latin       0.35      0.48      0.41      3670
                 pop       0.26      0.16      0.19      4387
              reggae       0.19      0.02      0.03       776
          rock_metal       0.41      0.57      0.47      6737
               world       0.21      0.02      0.03      3155

            accuracy                           0.41     35896
           macro avg       0.32      0.27      0.25     35896
        weighted avg       0.36      0.41      0.36     35896



### comparing C (regularization) F1 scores

In [11]:
for C in [0.01, 0.1, 1, 10]:
    model.set_params(C=C)
    model.fit(X_train_s, y_train)
    score = f1_score(y_val, model.predict(X_val_s), average='macro')
    print(f"C={C:<5} val macro-F1 = {score:.3f}")

C=0.01  val macro-F1 = 0.237
C=0.1   val macro-F1 = 0.250
C=1     val macro-F1 = 0.253
C=10    val macro-F1 = 0.253


C=1 (default) seems to have the higher scores. Will keep C as C=1 (default)